In [21]:
import pandas as pd
import yaml

intermediate_dir = 'output/intermediate/'
cfg = yaml.safe_load(open('configs/settings.yaml'))

In [2]:
# load CHAS HUD data
chas = (
    pd.read_excel('data/' + cfg['data']['hud_chas'], sheet_name='Data')
    .rename(columns={
        'Income by Cost Burden': 'income_by_cost_burden',
        'Cost burden > 30%  (Renters only)': 'cost_burden_over_30',
        'Total (Renters and Owners)': 'total_hh'
        })
)

chas['county_id'] = chas['County'].map(cfg['county_map'])


In [3]:
# Calculate total households by county from CHAS data
chas_tot_hh = chas.groupby('county_id')['total_hh'].sum()

# Use vacancy rates to calculate current OFM household counts
vac = pd.read_csv(intermediate_dir + '/vacancy_rates.csv').set_index('county_id')
ofm_units = pd.read_csv(intermediate_dir + 'ofm_housing_estimates.csv').set_index('county_id')
ofm_hh = round(ofm_units['total_housing_units'] * (1 - vac['vacancy_rate']), 0)

# calculate adjustment ratio by county to apply to CHAS data to grow to current OFM HH
adj_ratio = ofm_hh / chas_tot_hh


In [4]:
chas['adj_ratio'] = chas['county_id'].map(adj_ratio)

In [5]:
chas['cost_burden_over_30_adj'] = round(chas['cost_burden_over_30'] * chas['adj_ratio'], 0)

In [6]:
chas.groupby('county_id')['cost_burden_over_30'].sum()

county_id
53033    169335
53035     14890
53053     57510
53061     46055
Name: cost_burden_over_30, dtype: int64

In [7]:
chas.groupby('county_id')['cost_burden_over_30_adj'].sum()

county_id
53033    185623.0
53035     15827.0
53053     60620.0
53061     49303.0
Name: cost_burden_over_30_adj, dtype: float64

In [8]:
chas

,County,income_by_cost_burden,cost_burden_over_30,total_hh,county_id,adj_ratio,cost_burden_over_30_adj
0,King County,Household Income <= 30% HAMFI,61145,114475,53033,1.096188,67026.0
1,King County,Household Income >30% to <=50% HAMFI,47510,91390,53033,1.096188,52080.0
2,King County,Household Income >50% to <=80% HAMFI,39510,117325,53033,1.096188,43310.0
3,King County,Household Income >80% to <=100% HAMFI,12285,84505,53033,1.096188,13467.0
4,King County,Household Income >100% HAMFI,8885,508575,53033,1.096188,9740.0
5,Kitsap County,Household Income <= 30% HAMFI,4855,10770,53035,1.063017,5161.0
6,Kitsap County,Household Income >30% to <=50% HAMFI,3685,9360,53035,1.063017,3917.0
7,Kitsap County,Household Income >50% to <=80% HAMFI,4450,18615,53035,1.063017,4730.0
8,Kitsap County,Household Income >80% to <=100% HAMFI,1325,12090,53035,1.063017,1408.0
9,Kitsap County,Household Income >100% HAMFI,575,55200,53035,1.063017,611.0


In [15]:
import numpy as np

# CHAS Table 18C: renter-occupied units by unit affordability (rent <=30, 30-50, 50-80, >80 HAMFI) and occupant income level
t18c = pd.read_csv('data/' + cfg['data']['hud_chas_t18c'], dtype={'st': str, 'cnty': str})
t18c = t18c.set_index((t18c['st'] + t18c['cnty']).astype(int).rename('county_id'))

def unit_afford_matrix(row):
    """Return 4x5 array: rows = unit affordability level, cols = occupant income level."""
    m = np.zeros((4, 5))
    for s in (2, 27, 52, 77):  # one est column block per structure type
        for a in range(4):
            for k in range(5):
                m[a, k] += row[f'T18C_est{s + 2 + 6 * a + k}']
    return m

unit_afford = {cid: unit_afford_matrix(t18c.loc[cid]) for cid in cfg['county_ids']}
unit_afford[53033]

array([[ 24655.,   4405.,   2720.,   1905.,   3240.],
       [ 15610.,  12920.,   9930.,   4155.,   7305.],
       [ 26190.,  28620.,  32800.,  19990.,  39665.],
       [ 14205.,  10030.,  18110.,  14985., 109885.]])

In [16]:
levels = ['0-30%', '>30-50%', '>50-80%', '>80-100%', '>100-120%']

def move_out_probs(m):
    """Share of a moving cost-burdened household's vacated units by unit level (4 levels).

    Cost-burdened households are assumed to live in units priced above their income band.
    """
    p = np.zeros((5, 4))
    for i in range(5):
        lowest = min(min(i, 3) + 1, 3)
        allowed = np.arange(4) >= lowest
        counts = m[:, i] * allowed
        p[i] = counts / counts.sum()
    return p

def vacancy_chain(cb_hh, m, vac_rate, years=25):
    """Build 1/25th of each level's need per year; vacated units are re-occupied until no cost burden remains."""
    p = move_out_probs(m)
    occ = 1 - vac_rate
    need = np.asarray(cb_hh, dtype=float).copy()
    annual = need / occ / years
    new_prod, vacated = np.zeros(5), np.zeros(5)

    for _ in range(years):
        build = np.minimum(annual, need / occ)
        new_prod += build
        need -= build * occ
        movers = build * occ
        for _ in range(500):
            if movers.sum() < 1e-9:
                break
            released = movers @ p  # units vacated by unit level
            # units above 80% AMI are split between the 80-100 and 100-120 bands by remaining need
            top_need = need[3:]
            top_share = top_need / top_need.sum() if top_need.sum() > 0 else np.zeros(2)
            avail = np.concatenate([released[:3], released[3] * top_share])
            housed = np.minimum(avail * occ, need)
            need -= housed
            vacated += housed / occ
            movers = housed
    return new_prod, vacated

results = []
for cid in cfg['county_ids']:
    cb = chas.loc[chas['county_id'] == cid, 'cost_burden_over_30_adj'].to_numpy()
    prod, vacated = vacancy_chain(cb, unit_afford[cid], vac.loc[cid, 'vacancy_rate'])
    results.append(pd.DataFrame({
        'county_id': cid, 'income_level': levels, 'cb_hh_2020': cb,
        'new_production': prod.round(), 'units_vacated': vacated.round(),
    }))
cb_need = pd.concat(results, ignore_index=True)
cb_need

,county_id,income_level,cb_hh_2020,new_production,units_vacated
0,53033,0-30%,67026.0,70785.0,0.0
1,53033,>30-50%,52080.0,41550.0,13451.0
2,53033,>50-80%,43310.0,16466.0,29273.0
3,53033,>80-100%,13467.0,569.0,13653.0
4,53033,>100-120%,9740.0,411.0,9875.0
5,53035,0-30%,5161.0,5524.0,0.0
6,53035,>30-50%,3917.0,3019.0,1174.0
7,53035,>50-80%,4730.0,2228.0,2835.0
8,53035,>80-100%,1408.0,60.0,1447.0
9,53035,>100-120%,611.0,26.0,628.0


In [17]:
# Exhibit 22: housing needed for the population experiencing homelessness
summary = pd.read_excel('output/housing_need.xlsx', sheet_name='summary')
units_col = next(c for c in summary.columns if c.startswith('OFM Current Housing Units'))
need_col = f"Est. Total Housing Need {cfg['forecast_year']}"
homeless_total = summary.loc[summary['County'] == 'Region', 'Est. Current Shortage from Homelessness'].iloc[0]

county_sum = summary[summary['County'].isin(cfg['county_map'])].copy()
county_sum['county_id'] = county_sum['County'].map(cfg['county_map'])
county_sum = county_sum.set_index('county_id')

# Regional homelessness is allocated to counties by share of current housing units
hl = pd.DataFrame({'ofm_units': county_sum[units_col]})
hl['hh_homeless'] = homeless_total * hl['ofm_units'] / hl['ofm_units'].sum()
hl['0-30%'] = (hl['hh_homeless'] * 0.9).round()
hl['>30-50%'] = (hl['hh_homeless'] * 0.1).round()
hl

,ofm_units,hh_homeless,0-30%,>30-50%
county_id,,,,
53033,1060733.0,22346.872632,20112.0,2235.0
53035,120648.0,2541.738109,2288.0,254.0
53053,381090.0,8028.570518,7226.0,803.0
53061,345122.0,7270.818742,6544.0,727.0


In [18]:
# Exhibit 23: additional housing to accommodate population growth
cb_prod = cb_need.groupby('county_id')['new_production'].sum()
ex23 = pd.DataFrame({
    'total_net_new': county_sum[need_col] - county_sum[units_col],
    'cost_burden_prod': cb_prod,
    'homeless': hl[['0-30%', '>30-50%']].sum(axis=1),
})
ex23['growth'] = (ex23['total_net_new'] - ex23['cost_burden_prod'] - ex23['homeless']).clip(lower=0)
# If baseline needs exceed the total, the total is raised to match and there is no growth component
ex23['total_net_new_adj'] = ex23[['cost_burden_prod', 'homeless', 'growth']].sum(axis=1)
ex23.round()

,total_net_new,cost_burden_prod,homeless,growth,total_net_new_adj
county_id,,,,,
53033,578527.0,129781.0,22347.0,426399.0,578527.0
53035,29438.0,10857.0,2542.0,16039.0,29438.0
53053,150905.0,42465.0,8029.0,100411.0,150905.0
53061,203618.0,35244.0,7271.0,161103.0,203618.0


In [22]:
# Exhibit 24: share of households by income level
# CHAS lumps all >100% HAMFI together, so PUMS is used to split it into >100-120% and >120%
import io
import zipfile

def read_hud_il(path):
    """Read a HUD Income Limits workbook, dropping its core-properties part (malformed dates break openpyxl)."""
    buf = io.BytesIO()
    with zipfile.ZipFile(path) as zin, zipfile.ZipFile(buf, 'w') as zout:
        for item in zin.infolist():
            if item.filename != 'docProps/core.xml':
                zout.writestr(item, zin.read(item.filename))
    buf.seek(0)
    return pd.read_excel(buf)

# FY2024 median family income (4-person) matches the 2024 dollars of PUMS incomes
hud = read_hud_il('data/' + cfg['data']['hud_income_limits'])
hud = hud[hud['stusps'] == 'WA'].assign(county_id=lambda d: d['fips'] // 100000).set_index('county_id')
ami4 = hud['median2024']

size_factor = {1: 0.70, 2: 0.80, 3: 0.90, 4: 1.0, 5: 1.08, 6: 1.16, 7: 1.24, 8: 1.32}
puma_cnty = pd.read_csv(intermediate_dir + 'puma_geog_lookup_cnty.csv')[['PUMA', 'county_id']]
pums = (
    pd.read_csv(intermediate_dir + 'psam_h53.csv', usecols=['PUMA', 'HINCP', 'ADJINC', 'NP', 'WGTP', 'TYPEHUGQ'])
    .query('TYPEHUGQ == 1 and NP > 0 and WGTP > 0')
    .dropna(subset=['HINCP'])
    .merge(puma_cnty, on='PUMA')
)
hamfi = pums['NP'].clip(upper=8).map(size_factor) * pums['county_id'].map(ami4)
pums['ratio'] = pums['HINCP'] * pums['ADJINC'] / 1e6 / hamfi

def share_100_120_of_over_100(df):
    """Share of households above 100% HAMFI that are at or below 120%."""
    r, w = df['ratio'].to_numpy(), df['WGTP'].to_numpy()
    return w[(r > 1) & (r <= 1.2)].sum() / w[r > 1].sum()

hh_by_inc = chas.pivot_table(index='county_id', columns=chas.groupby('county_id').cumcount(), values='total_hh')
levels6 = levels + ['>120%']
pct = pd.DataFrame(index=hh_by_inc.index, columns=levels6, dtype=float)
for cid, row in hh_by_inc.iterrows():
    total_hh = row.sum()
    r120 = share_100_120_of_over_100(pums[pums['county_id'] == cid])
    over100 = row.iloc[4] / total_hh
    pct.loc[cid] = [*(row.iloc[:4] / total_hh), over100 * r120, over100 * (1 - r120)]
(pct * 100).round(2)

,0-30%,>30-50%,>50-80%,>80-100%,>100-120%,>120%
county_id,,,,,,
53033,12.49,9.97,12.80,9.22,8.52,46.98
53035,10.16,8.83,17.56,11.40,10.62,41.44
53053,9.87,9.92,17.78,11.53,10.43,40.47
53061,12.08,11.66,15.81,11.55,9.95,38.95


In [23]:
# Exhibit 25: total net new housing need by income level
cb_by_level = cb_need.pivot(index='county_id', columns='income_level', values='new_production').reindex(columns=levels)
cb_by_level['>120%'] = 0.0
hl_by_level = pd.DataFrame(0.0, index=cb_by_level.index, columns=levels6)
hl_by_level[['0-30%', '>30-50%']] = hl[['0-30%', '>30-50%']]
growth_by_level = pct.mul(ex23['growth'], axis=0)

rows = []
for cid in cb_by_level.index:
    df = pd.DataFrame({
        'cost_burden': cb_by_level.loc[cid], 'homeless': hl_by_level.loc[cid], 'growth': growth_by_level.loc[cid],
    })
    df['total'] = df.sum(axis=1)
    df.loc['Total'] = df.sum()
    df.insert(0, 'county_id', cid)
    rows.append(df)
ex25 = pd.concat(rows).round()
ex25

,county_id,cost_burden,homeless,growth,total
0-30%,53033,70785.0,20112.0,53273.0,144170.0
>30-50%,53033,41550.0,2235.0,42530.0,86315.0
>50-80%,53033,16466.0,0.0,54599.0,71065.0
>80-100%,53033,569.0,0.0,39326.0,39895.0
>100-120%,53033,411.0,0.0,36334.0,36745.0
>120%,53033,0.0,0.0,200338.0,200338.0
Total,53033,129781.0,22347.0,426399.0,578527.0
0-30%,53035,5524.0,2288.0,1629.0,9441.0
>30-50%,53035,3019.0,254.0,1416.0,4689.0
>50-80%,53035,2228.0,0.0,2816.0,5044.0


In [24]:
# Export each exhibit to its own tab with county names and readable headings
from openpyxl.styles import Alignment, Font
from openpyxl.utils import get_column_letter

county_names = {v: k for k, v in cfg['county_map'].items()}
INC = 'Income Level (% of AMI)'

def add_region_total(df):
    total = df.drop(columns='County').sum()
    return pd.concat([df, pd.DataFrame([{'County': 'Region', **total}])], ignore_index=True)

# Exhibit 21: housing needed to eliminate renter cost burden
raw_cb = chas.assign(income_level=chas.groupby('county_id').cumcount().map(dict(enumerate(levels))))[
    ['county_id', 'income_level', 'cost_burden_over_30']
]
t21 = cb_need.merge(raw_cb, on=['county_id', 'income_level'])
t21.insert(0, 'County', t21['county_id'].map(county_names))
t21 = t21[['County', 'income_level', 'cost_burden_over_30', 'cb_hh_2020', 'new_production', 'units_vacated']]
t21.columns = [
    'County', INC, 'Cost-Burdened Renter Households (CHAS)', 'Cost-Burdened Renter Households (Scaled to OFM Estimates)',
    'New Production to Address Need', 'Units Vacated That Address Need',
]
parts = []
for county, g in t21.groupby('County', sort=False):
    total = {'County': county, INC: 'Total', **g.drop(columns=['County', INC]).sum()}
    parts += [g, pd.DataFrame([total])]
t21 = pd.concat(parts, ignore_index=True)

# Exhibit 22: housing needed for the population experiencing homelessness
t22 = hl.reset_index()
t22.insert(0, 'County', t22.pop('county_id').map(county_names))
t22 = add_region_total(t22)
t22.columns = [
    'County', 'Current Housing Units (OFM)', 'Households Experiencing Homelessness (Allocated)',
    'Housing Needed at 0-30% of AMI (90%)', 'Housing Needed at >30-50% of AMI (10%)',
]

# Exhibit 23: additional housing to accommodate population growth
t23 = ex23.reset_index()
t23.insert(0, 'County', t23.pop('county_id').map(county_names))
t23 = add_region_total(t23)
t23.columns = [
    'County', 'Total Projected Net New Units Needed', 'Housing to Eliminate Baseline Renter Cost Burden',
    'Housing for Baseline Homeless Population', 'Additional Housing to Accommodate Population Growth',
    'Total Net New Units Needed (Adjusted)',
]

# Exhibit 24: share of households by income level
t24 = pct.reset_index()
t24.insert(0, 'County', t24.pop('county_id').map(county_names))
t24.columns = ['County'] + [f'{lv} of AMI' for lv in levels6]

# Exhibit 25: total net new housing need by income level
t25 = ex25.reset_index(names=INC)
t25.insert(0, 'County', t25.pop('county_id').map(county_names))
t25.columns = [
    'County', INC, 'Housing Production Needed to Eliminate Renter Cost Burden', 'Housing Needed for Population Experiencing Homelessness',
    'Additional Housing to Accommodate Population Growth', 'Total Net New Housing Needed',
]

sheets = {
    'Cost Burden Housing Need': (t21, '#,##0'),
    'Homelessness Housing Need': (t22, '#,##0'),
    'Population Growth Housing Need': (t23, '#,##0'),
    'Household Income Shares': (t24, '0.0%'),
    'Total Need by Income Level': (t25, '#,##0'),
}

out_path = 'output/housing_need_by_income.xlsx'
with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
    for name, (df, num_fmt) in sheets.items():
        df.to_excel(writer, sheet_name=name, index=False)
        ws = writer.sheets[name]
        ws.row_dimensions[1].height = 48
        for i, col in enumerate(df.columns, start=1):
            letter = get_column_letter(i)
            header = ws[f'{letter}1']
            header.font = Font(bold=True)
            header.alignment = Alignment(wrap_text=True, vertical='center', horizontal='center')
            if pd.api.types.is_numeric_dtype(df[col]):
                ws.column_dimensions[letter].width = 22
                for cell in ws[letter][1:]:
                    cell.number_format = num_fmt
            else:
                ws.column_dimensions[letter].width = 24
        ws.freeze_panes = 'A2'
print('Saved', out_path)

Saved output/housing_need_by_income.xlsx
